# N039 · 快速排序与划分不变量

**目标：** 保持划分区域含义并处理重复元素。

**先修：** N037, N038。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Lomuto/Hoare区别；三路划分；随机枢轴；最坏复杂度；递归深度。

**配套讲解来源：** [同名逐章意见](../../comment/039_quicksort_partition.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的问题是：**怎么写一个"快"的排序——快速排序**，并且要专门对付两个坑：第一，数组里有**大量重复元素**时，朴素的二路快排会把相等元素反复送进下一层递归，白白浪费；第二，Python 的递归深度有上限（默认约 1000 层），快排最坏情况递归深度是 O(n)，大数组直接抛 RecursionError。

先看具体到数字的例子（“运行示例”部分 的真实数据）：

- 输入：`a = [3,1,3,5,2,3,4]`，对它调用 `partition3(a, 0, 7)`（以 3 为枢轴做一次三路划分）
- 输出：返回等值块下标 `(lt, gt) = (2, 5)`，数组被原地重排成 `[1,2,3,3,3,4,5]`
- 为什么？划分要达到的效果是：枢轴 3 左边的段全部严格小于 3，中间的段全部等于 3，右边的段全部严格大于 3。划分后 a[:2]=[1,2]（<3），a[2:5]=[3,3,3]（=3），a[5:]=[4,5]（>3）。注意那个"等于 3 的块"一次就吞掉了全部三个 3——这就是三路划分对重复值的杀伤力：这些 3 从此不再进入任何递归。完整的 `quicksort` 递归处理两侧，最终得到有序数组 [1,2,3,3,3,4,5]。

## 2. 基础知识：先读懂这些词

- **划分（partition）**：选定一个基准值 pivot，把数组重排成"小于 pivot 的都放左边、等于的放中间、大于的放右边"的一次操作。它是快排的基本动作。
- **Lomuto / Hoare 划分**：两种经典二路划分写法（单向扫描 / 双向夹逼）。它们只分"小于等于 / 大于"或"小于 / 大于等于"两段，遇到全相等的数组会退化。本章不用它们，改用三路。
- **三路划分（three-way partition）**：把数组分成三段（小于、等于、大于 pivot），又称荷兰国旗问题。它的好处是：所有等于 pivot 的元素一次归位，不再进入后续递归——重复元素越多，省得越多。
- **枢轴（pivot）**：划分的基准值。本章取 `a[lo]`（先把随机选中的元素换到 lo 位置）。
- **四段不变量**：划分过程中数组始终是四段——`[lo:lt)` 全部小于 pivot，`[lt:i)` 全部等于 pivot，`[i:gt)` 是尚未分类的元素，`[gt:hi)` 全部大于 pivot。循环每走一步都在维持这句话。
- **随机枢轴（random pivot）**：划分前从当前区间随机挑一个元素当枢轴，让"每次都挑到最差枢轴"这种倒霉事变成小概率事件。它不改变排序对不对，只改变快慢的分布。
- **显式栈（explicit stack）**：用一个普通 list 模拟递归的"待办清单"，代替 Python 的函数递归。栈放在堆内存上，不受递归深度限制，这是本章避开 RecursionError 的手段。
- **递归深度**：递归调用的嵌套层数。快排每次划分出两段继续处理，若总是极度偏斜（一段空、一段几乎全长），深度就到 O(n)。

## 3. 思路推导：从小例子开始

- **Step 1：选枢轴并换到开头。** 在当前未排序区间 [lo, hi) 里随机挑一个下标 p，把 a[p] 与 a[lo] 交换。此后 `partition3` 统一用 a[lo] 当枢轴。
- **Step 2：初始化四段。** pivot = a[lo]；三根指针 lt = lo（小于段的右边界）、i = lo（当前待分类元素）、gt = hi（大于段的左边界）。初始时"小于段""等于段""大于段"都是空的，"未处理段"是整个区间。
- **Step 3：逐个分类 a[i]，三分支循环。**
  - a[i] < pivot：把它换到 lt 处（那里是等于段的头），lt 前进一格，i 前进一格。换过来的元素原本就属于等于段，已分类，i 可以放心前进。
  - a[i] > pivot：gt 先退一格，把 a[i] 换到 gt 处。**换过来的元素来自未处理段的末尾，还没分类，所以 i 不能前进**——下一轮循环还要再给它分类。
  - a[i] == pivot：它已在等于段的位置上，i 前进一格即可。
- **Step 4：收尾。** i 与 gt 相遇时未处理段清空，返回 (lt, gt)：闭开区间 [lt, gt) 就是等于块。
- **Step 5：递归（用栈实现）。** 把 (lo, lt) 和 (gt, hi) 两个区间压栈继续处理；等于块已经最终归位，不再管。

**手算演示：`partition3` 处理 `[3,1,3,5,2,3,4]`（“运行示例”部分 的数据，一格一格走）。** pivot = a[0] = 3，lt = i = 0，gt = 7：

| 步骤 | 看到的元素 | 动作 | 数组变为 | lt / i / gt |
|---|---|---|---|---|
| 1 | a[0]=3 == 3 | i 前进 | [3,1,3,5,2,3,4] | 0 / 1 / 7 |
| 2 | a[1]=1 < 3 | 换到 lt，双双前进 | [1,3,3,5,2,3,4] | 1 / 2 / 7 |
| 3 | a[2]=3 == 3 | i 前进 | [1,3,3,5,2,3,4] | 1 / 3 / 7 |
| 4 | a[3]=5 > 3 | gt 退一格，换到 gt；**i 不动** | [1,3,3,4,2,3,5] | 1 / 3 / 6 |
| 5 | a[3]=4 > 3 | 同上，**i 仍不动** | [1,3,3,3,2,4,5] | 1 / 3 / 5 |
| 6 | a[3]=3 == 3 | i 前进 | [1,3,3,3,2,4,5] | 1 / 4 / 5 |
| 7 | a[4]=2 < 3 | 换到 lt，双双前进 | [1,2,3,3,3,4,5] | 2 / 5 / 5 |

此时 i = 5 = gt，未处理段清空，返回 (lt, gt) = (2, 5)。对着结果检查四段：a[:2]=[1,2] 全小于 3；a[2:5]=[3,3,3] 全等于 3；a[5:]=[4,5] 全大于 3——正是 “运行示例”部分 那张表给出的三段。步骤 4、5 连续两次"大于"处理是这段演示最值得回看的地方：换回来的 4、3 还没分类，i 原地等下一轮，直到步骤 6 才把换回来的 3 认领进等于段。

**接着看 quicksort 怎么用栈收尾（接上面的 `[1,2,3,3,3,4,5]`、lt=2、gt=5）。** 划分结束，栈里压入 (0,2) 和 (5,7) 两段，等值块 [2,5) 不再处理：

1. 弹出 (5,7)：段内是 [4,5]。假设随机选中下标 6 的值 5 当枢轴，换到段首后划分：5 等于枢轴 i 前进；4 小于枢轴换到 lt。返回等值块 (6,7)，那个 5 就此归位。再压入 (5,6)、(7,7)。
2. 弹出 (7,7) 和 (5,6)：长度都小于 2，`continue` 跳过——零个或一个元素天然有序。
3. 弹出 (0,2)：段内是 [1,2]。假设随机选中 1 当枢轴：1 等于枢轴 i 前进；2 大于枢轴 gt 左移自换。返回等值块 (0,1)，那个 1 归位。压入 (0,0)、(1,2)。
4. 弹出 (1,2)：长度 1，跳过。栈空收工，数组恰为 [1,2,3,3,3,4,5]。

看清楚整条流水线：每次划分把"等于枢轴的一整块"钉死在最终位置，栈只负责记账还没处理的两段——递归能做的事，一个 list 加 while 循环一样能做，还躲开了 Python 的递归深度上限。

## 4. 核心代码：partition3

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def partition3(a, lo, hi):
    """Partition half-open [lo, hi); return equal block [lt, gt)."""
    if not 0 <= lo < hi <= len(a):
        raise ValueError('requires nonempty valid range')
    pivot = a[lo]
    lt = i = lo
    gt = hi
    while i < gt:
        if a[i] < pivot:
            a[lt], a[i] = (a[i], a[lt])
            lt += 1
            i += 1
        elif a[i] > pivot:
            gt -= 1
            a[gt], a[i] = (a[i], a[gt])
        else:
            i += 1
    return (lt, gt)
```

### 逐段读懂代码

### 函数 1：`partition3` —— 三路划分

```python
def partition3(a, lo, hi):
    """Partition half-open [lo, hi); return equal block [lt, gt)."""
    if not 0 <= lo < hi <= len(a):
        raise ValueError('requires nonempty valid range')
    pivot = a[lo]
    lt = i = lo
    gt = hi
    while i < gt:
        if a[i] < pivot:
            a[lt], a[i] = (a[i], a[lt])
            lt += 1
            i += 1
        elif a[i] > pivot:
            gt -= 1
            a[gt], a[i] = (a[i], a[gt])
        else:
            i += 1
    return (lt, gt)
```

- 文档字符串：说明区间约定是半开 [lo, hi)（含 lo 不含 hi），返回等值块 [lt, gt)。半开约定让"空区间"就是 lo==hi，判断起来干净。
- `if not 0<=lo<hi<=len(a): raise ValueError(...)`：输入守卫，链式比较一次查三件事——lo 不小于 0、lo 小于 hi（区间非空）、hi 不越界。空区间或非法下标直接报错，因为划分对空区间没有意义（quicksort 那边用 `hi-lo<2` 提前跳过了，永远不会拿空区间进来）。
- `pivot=a[lo]; lt=i=lo; gt=hi`：枢轴取区间头；lt 是"小于段"的右边界，i 是当前待分类位置，gt 是"大于段"的左边界。`lt=i=lo` 是链式赋值。
- `if a[i]<pivot:` 分支：把 a[i] 换到 lt 位置（等于段的开头），然后 lt 和 i 各前进一格。换到 i 位置的元素来自等于段，已经分好类，i 放心走。
- `elif a[i]>pivot:` 分支：注意语句顺序——先 `gt-=1` 再交换，把 a[i] 送到新 gt 的位置。换回来的元素来自未处理段尾部，分类情况未知，**这个分支里没有 i+=1**，下一轮循环继续审它。
- `else: i+=1`：等于枢轴，就地归入等于段。
- `return lt,gt`：循环结束时 [lt, gt) 恰是等于块，调用方只需递归它两侧。

三个分支一张速查表（都在"当前看 a[i]"这个前提下）：

| 分支条件 | 动作 | i 前进吗 | 为什么 |
|---|---|---|---|
| a[i] < pivot | 换到 lt 处，lt、i 各 +1 | 前进 | 换回来的是等于段的元素，已分类 |
| a[i] > pivot | gt 先 −1，再换到 gt 处 | 不动 | 换回来的元素来自未处理段，还要再审 |
| a[i] == pivot | 不交换 | 前进 | 它已经在等于段右端就位 |

### 函数 2：`quicksort` —— 显式栈版快排

```python
def quicksort(a, rng):
    stack = [(0, len(a))]
    while stack:
        lo, hi = stack.pop()
        if hi - lo < 2:
            continue
        p = rng.randrange(lo, hi)
        a[lo], a[p] = (a[p], a[lo])
        lt, gt = partition3(a, lo, hi)
        stack.append((lo, lt))
        stack.append((gt, hi))
    return a
```

- `quicksort(a,rng)`：第二个参数是随机数发生器，由调用方传入（测试里传 `Random(7)` 固定种子，保证可复现）。随机源不藏在函数内部，这是可测性的好习惯。
- `stack=[(0,len(a))]`：待处理区间清单，初始只有整个数组。用 list 当栈，pop 从尾部取，后进先出。
- `if hi-lo<2: continue`：区间里 0 或 1 个元素天然有序，跳过。这一行同时保证了传给 partition3 的区间非空，不会触发表里的 ValueError。
- `p=rng.randrange(lo,hi); a[lo],a[p]=a[p],a[lo]`：在区间内随机挑下标 p，把它换到开头，让 partition3 拿 a[lo] 当枢轴。随机化只影响"枢轴质量"，不影响划分的正确性。
- `lt,gt=partition3(a,lo,hi)`：三路划分，等值块 [lt, gt) 已最终就位。
- `stack.append((lo,lt)); stack.append((gt,hi))`：把"小于段"和"大于段"两段压栈，等下一轮循环处理。等于段**不压栈**——这就是三路划分对付重复元素的全部秘密。整个过程没有函数自我调用，深度由 list 长度决定，不怕 Python 递归上限。
- `return a`：原地排序完成，返回原列表。

## 5. 分段实现：按顺序运行

**本章接口：** `partition3(a, lo, hi)`、`quicksort(a, rng)`

### partition3

In [1]:
def partition3(a, lo, hi):
    """Partition half-open [lo, hi); return equal block [lt, gt)."""
    if not 0 <= lo < hi <= len(a):
        raise ValueError('requires nonempty valid range')
    pivot = a[lo]
    lt = i = lo
    gt = hi
    while i < gt:
        if a[i] < pivot:
            a[lt], a[i] = (a[i], a[lt])
            lt += 1
            i += 1
        elif a[i] > pivot:
            gt -= 1
            a[gt], a[i] = (a[i], a[gt])
        else:
            i += 1
    return (lt, gt)

### quicksort

In [2]:
def quicksort(a, rng):
    stack = [(0, len(a))]
    while stack:
        lo, hi = stack.pop()
        if hi - lo < 2:
            continue
        p = rng.randrange(lo, hi)
        a[lo], a[p] = (a[p], a[lo])
        lt, gt = partition3(a, lo, hi)
        stack.append((lo, lt))
        stack.append((gt, hi))
    return a

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,1,3,5,2,3,4]; original=a[:]; lt,gt=partition3(a,0,len(a))
show_table(['原数组','<pivot','=pivot','>pivot'],[(original,a[:lt],a[lt:gt],a[gt:])])

原数组,<pivot,=pivot,>pivot
"[3, 1, 3, 5, 2, 3, 4]","[1, 2]","[3, 3, 3]","[4, 5]"


## 7. 正确性、适用条件与复杂度

每次循环使未处理区长度减少1，并维持已分类区关系；划分后只需排序小于和大于两段。随机pivot不改变正确性，只影响复杂度分布。

**代价：** 随机独立pivot下期望 O(n log n)，最坏 O(n²)；显式栈最坏 O(n)，原地划分 O(1)。

### 展开理解

**为什么 partition3 一定停、停得对？** 我们把循环里那张"四段图"用完整的话说一遍。任意一轮循环开始时，数组都满足这句话：从 lo 到 lt 的元素全部严格小于枢轴；从 lt 到 i 的元素全部等于枢轴；从 i 到 gt 是还没看过的元素；从 gt 到 hi 的元素全部严格大于枢轴。开头（第一轮之前）：三根指针把区间切成三个空段加一个全未处理段，这句话自动成立。每走一轮：小于分支把一个元素送进小于段、lt 和 i 同时右移，等于段没有混入新东西；大于分支把一个元素送进大于段、gt 左移，未处理段从右端缩一格；等于分支让等于段右端长一格、未处理段左端缩一格——无论哪个分支，未处理段 [i, gt) 的长度都严格减少，所以循环最多走 (hi-lo) 轮必然停下；停的时候未处理段是空的，四段图就坍缩成"小于段、等于段、大于段"三段，正是我们要的结论。这就是 “正确性与复杂度”部分 说的"每次循环使未处理区长度减少 1，并维持已分类区关系"。

**为什么 quicksort 整体有序？** 每次划分之后，等于块里的元素已经站在它们最终该在的位置上（左边全比它们小、右边全比它们大），所以只需继续处理两侧两段；两段又各自重复同样的论证，段越切越小，最后每段都小于 2 个元素，全部元素各就各位。随机交换枢轴只是换了"用哪个值当基准"，四段图的论证一字不改，所以 “正确性与复杂度”部分 说"随机 pivot 不改变正确性，只影响复杂度分布"。

**复杂度是多少？** 划分本身线性、原地（只多花 O(1) 个变量）。整体快慢取决于切得多匀：随机枢轴下期望每层区间缩到原来的某个比例，期望总代价 O(n log n)；最坏（每次枢轴都是当前段的最值）退化成 O(n²)。拿数字感受：n = 10⁵ 时，期望约 170 万次操作量级，一瞬间；最坏情形约 10¹⁰ 次，等不起——但随机化让"攻击者构造最坏输入"变成几乎不可能的事。显式栈最坏要存 O(n) 个区间（list 里的元组），花的是堆内存不是递归栈；练习 2 会讨论怎么把它压到 O(log n)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看（随机源 `rng=Random(39)`、n 从 0 到 79，每个长度造一个取值 −3..3 的随机数组——取值只有 7 种，保证重复元素成堆）：

- `assert quicksort(a[:],Random(7))==sorted(a)`：**正常值测试 + 交叉验证（排序结果 vs 内置 sorted）**。注意两处细节：传 `a[:]` 复制，因为 quicksort 是原地排序，不复制会破坏 a 供后面断言用；传 `Random(7)` 固定种子，让枢轴选择可复现，测试不抖。
- `assert all(x<pivot for x in b[:lt]) and all(x==pivot for x in b[lt:gt]) and all(x>pivot for x in b[gt:])`：**划分后置条件测试**。对单独调用一次 partition3 的结果，逐段检查"小于段全严格小于、等于段全相等、大于段全严格大于"。这测的是划分本身的契约，与排序无关。
- `assert Counter(a)==Counter(b)`：**元素守恒测试（多重集合不变）**。排序/划分只许重排、不许丢元素或克隆元素；用 Counter 数一遍重排前后的元素出现次数，必须完全一致。这能抓住"交换写错导致元素被覆盖"一类 bug。
- `assert quicksort([2]*1000,Random(1))==[2]*1000`：**极端值压力测试（全部元素相等）**。1000 个相同的 2 正是朴素二路快排的噩梦（每层只切掉一个元素）；三路划分一次就把 1000 个全吞进等值块，一轮结束。这条既验证正确性，又展示三路划分的卖点。

| 断言 | 类型 | 它防的是哪类 bug |
|---|---|---|
| quicksort vs sorted（80 个随机数组） | 交叉验证 | 整体排序结果错 |
| all(...) 三段后置条件 | 划分契约 | 某段混入了不合格元素 |
| Counter(a)==Counter(b) | 元素守恒 | 交换写错导致元素被覆盖丢失 |
| quicksort([2]*1000) | 极端压力 | 重复值输入下退化或答错 |

In [4]:
from random import Random

from collections import Counter

rng = Random(39)

for n in range(80):
    a = [rng.randrange(-3, 4) for _ in range(n)]
    assert quicksort(a[:], Random(7)) == sorted(a)
    if a:
        b = a[:]
        pivot = b[0]
        lt, gt = partition3(b, 0, n)
        assert all((x < pivot for x in b[:lt])) and all((x == pivot for x in b[lt:gt])) and all((x > pivot for x in b[gt:]))
        assert Counter(a) == Counter(b)

assert quicksort([2] * 1000, Random(1)) == [2] * 1000

print('N039: 所有本章断言通过')

N039: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造固定枢轴退化输入。

**练习 2：** 改为先递归较短区间。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（构造固定枢轴的退化输入）**：方向提示——把随机挑枢轴改成"永远用 a[lo]"（或 a[hi]），然后喂一个已经升序的数组 [1,2,…,n]。手推每轮：枢轴是当前段最小值，划分后小于段为空、大于段少一个元素，一共要划 n 轮，总比较次数 1+2+…+n ≈ n²/2——n=1000 就是约 50 万次、栈深 n 层，退化肉眼可见。写报告时把"每轮区间长度只减 1"这句关键观察放进去，并用本章接口（改一个固定枢轴版本）实测对比随机版的轮数。
- **练习 2（改为先递归较短区间）**：方向提示——这是压栈深的经典技巧：先处理较短的那段、把较长段压栈（或先递归短段再迭代长段）。为什么有用？因为较短段长度不超过当前段的一半，所以任何时刻栈里"还没展开的长段"每层至多一个、长度逐层至少减半，栈的大小被 O(log n) 封顶，即使遇到最坏划分也不爆栈。手算示例可以用 [1,2,3,4]（固定枢轴 a[lo] 的最坏输入）对比两种压栈顺序下栈的最大长度。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def partition3(a, lo, hi):
    """Partition half-open [lo, hi); return equal block [lt, gt)."""
    if not 0 <= lo < hi <= len(a):
        raise ValueError('requires nonempty valid range')
    pivot = a[lo]
    lt = i = lo
    gt = hi
    while i < gt:
        if a[i] < pivot:
            a[lt], a[i] = (a[i], a[lt])
            lt += 1
            i += 1
        elif a[i] > pivot:
            gt -= 1
            a[gt], a[i] = (a[i], a[gt])
        else:
            i += 1
    return (lt, gt)

def quicksort(a, rng):
    stack = [(0, len(a))]
    while stack:
        lo, hi = stack.pop()
        if hi - lo < 2:
            continue
        p = rng.randrange(lo, hi)
        a[lo], a[p] = (a[p], a[lo])
        lt, gt = partition3(a, lo, hi)
        stack.append((lo, lt))
        stack.append((gt, hi))
    return a

# 示例与回归测试
from random import Random

from collections import Counter

rng = Random(39)

for n in range(80):
    a = [rng.randrange(-3, 4) for _ in range(n)]
    assert quicksort(a[:], Random(7)) == sorted(a)
    if a:
        b = a[:]
        pivot = b[0]
        lt, gt = partition3(b, 0, n)
        assert all((x < pivot for x in b[:lt])) and all((x == pivot for x in b[lt:gt])) and all((x > pivot for x in b[gt:]))
        assert Counter(a) == Counter(b)

assert quicksort([2] * 1000, Random(1)) == [2] * 1000

print('N039: 所有本章断言通过')

N039: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **912. Sort an Array（排序数组）**：这题练的就是把本章三路快排原样搬上去——随机枢轴满足期望复杂度要求，三路划分专门消化题里可能的大量重复值，显式栈躲开 Python 递归限制。可以与 N038 的归并版本同题对比：快排原地、常数小，归并稳定、最坏有保证。

照 notebook 原文提醒：以上条目用于知识映射，本章实现遵循教学契约，刷题时以 LeetCode 官方题面与评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。